In [2]:
import pandas as pd
import numpy as np
import json

In [3]:
import json
import os

# 设置路径
data_dir = "../eval/eval_results"
filename = "gsm8k_instruct_128_64_0_generations.json"  # 128 大小的文件
file_path = os.path.join(data_dir, filename)

# 检查文件是否存在
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

# 读取 JSON 文件
with open(file_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"Loaded {len(results)} samples from {filename}")


Loaded 7 samples from gsm8k_instruct_128_64_0_generations.json


In [9]:
import os
import json

# 配置
data_dir = "../eval/eval_results"  # 替换为你的实际路径
base_name = "gsm8k_instruct_128_64_{}_generations.json"
num_files = 8  # 从 0 到 7

all_generations = []

for i in range(num_files):
    filename = base_name.format(i)
    file_path = os.path.join(data_dir, filename)
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # 假设每个文件是 {'generations': [...]}
        if 'generations' not in data:
            raise KeyError(f"'generations' key not found in {filename}")
        all_generations.extend(data['generations'])

# 最终合并结果：all_generations 是一个大列表，包含所有样本
print(f"✅ Total samples loaded: {len(all_generations)}")

# 如果你希望保持和之前一样的结构（即 results = {'generations': [...] }）
results = {'generations': all_generations}

✅ Total samples loaded: 1319


In [4]:
def evaluate_accuracy(results):
    correct = 0
    total = len(results)
    
    for item in results:
        try:
            # 提取 ground_truth 和 generation
            gt = str(item['ground_truth']).strip()
            gen = str(item['generation']).strip()
            # extract_gen = extract_gsm8k_answer(gen)
            # 如果生成内容包含 ground_truth（宽松匹配）
            if gt in gen or gen == gt:
                correct += 1
            else:
                print(f"❌ Mismatch: GT={gt}, Gen={gen}")
        except Exception as e:
            print(f"Error processing item: {e}")
            continue

    accuracy = correct / total if total > 0 else 0
    return accuracy, correct, total

import re

def extract_answer(generations: str) -> str:
    """
    从模型生成的文本中提取 \boxed{...} 中的算术表达式。
    
    参数:
        generations (str): 模型生成的完整输出字符串
        
    返回:
        str: 提取出的表达式，如 "34 - 24 + 24"；若未找到则返回空字符串
    """
    # 使用正则表达式匹配 \boxed{...} 内容
    match = re.search(r'\\boxed\{([^}]*)\}', generations)
    if match:
        return match.group(1).strip()
    else:
        return ""

In [10]:
print(len(results))
print(results['generations'][1]['prompt_input'])
print(len(results['generations']))
print()
generated_texts = []
for i, sample in enumerate(results['generations']):
    prompt = sample['prompt_input']
    generated_text = sample['generations']  # ← 这才是你要的生成内容
    # 然后用 extract_gsm8k_answer(generated_text) 提取答案
    generated_texts.append(generated_text)

print(len(generated_texts))

1
<|startoftext|><|start_header_id|>user<|end_header_id|>

You are a math expert. You will be given a question to solve. Solve it step by step. Wrap the final answer in a \boxed{}. 
Respond in the following format:
<reasoning>
Your reasoning here
</reasoning>
<answer>
\boxed{...}
</answer>

John drives for 3 hours at a speed of 60 mph and then turns around because he realizes he forgot something very important at home.  He tries to get home in 4 hours but spends the first 2 hours in standstill traffic.  He spends the next half-hour driving at a speed of 30mph, before being able to drive the remaining time of the 4 hours going at 80 mph.  How far is he from home at the end of those 4 hours?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<reasoning>
1319

1319


In [11]:

def extract_gsm8k_answer(generations: str) -> float:
    """
    从 GSM8K 风格的模型输出中提取答案。
    
    策略：
    1. 优先尝试从 \\boxed{...} 中提取内容；
    2. 如果失败，则从整个 generations 文本中提取最后一个出现的数字（整数或浮点数）作为保底。

    参数:
        generations (str): 模型生成的回答字符串

    返回:
        float: 提取出的答案数值；若完全无法提取则返回 None
    """
    # Step 1: 尝试从 \boxed{...} 中提取
    boxed_match = re.search(r'\\boxed\{\s*([^\}]*)\s*\}', generations)
    if boxed_match:
        content = boxed_match.group(1).strip()
        # 移除可能的单位（如 "miles", "$", etc.）
        # 只保留数字、小数点、负号
        num_match = re.search(r'[-+]?(?:\d+\.?\d*|\.\d+)', content)
        if num_match:
            try:
                return float(num_match.group())
            except ValueError:
                pass  # fallback to step 2

    # Step 2: 保底策略 —— 提取整个文本中最后一个数字
    all_numbers = re.findall(r'[-+]?(?:\d+\.?\d*|\.\d+)', generations)
    if all_numbers:
        try:
            return float(all_numbers[-1])
        except ValueError:
            pass

    # 完全失败
    return None


In [7]:
# 运行评估
evals = []
print(generated_texts)
for text in generated_texts:
    eval = extract_gsm8k_answer(text)
    evals.append(eval)

print(evals)

["\nJanet's ducks lay 16 eggs per day. She eats 3 eggs for breakfast and bakes 4 eggs for her friends, which means she uses a total of \\(3 + 4 = 7\\) eggs per day. Therefore, she has \\(16 - 7 = 9\\) eggs left to sell. She sells each egg for $2, so she makes \\(9 \\times 2 = 18\\) dollars per day from selling the eggs.\n</reasoning>\n<answer>\n\\boxed{18}\n</answer><|eot_id|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>", '\nJohn drives for 3 hours at a speed of 60 mph, covering he covers 180 miles. He then turns around and drives to home home.  He spends 0.5 hours driving at a speed of 30 mph, covering 15 miles.  He spends 2 hours in standstill traffic, covering he covers 0 miles.  He then spends 1 hour driving at a speed of 80 mph, covering he covers 80 miles.\n</reasoning>\n<answer>\n\\boxed{15 \\text{ miles}}\n</answer><|eot_id|><|endoftext|>', '\nTo determine the total distance covered by each train over the 

In [12]:
# 假设 results 是你从模型生成得到的完整结果字典
evals = []
correct_count = 0
total = len(results['generations'])

for i, sample in enumerate(results['generations']):
    generated_text = sample['generations']
    ground_truth = sample['ground_truth']  # 确保这个字段存在！

    pred = extract_gsm8k_answer(generated_text)
    evals.append(pred)

    # 判断是否正确
    is_correct = False
    if pred is not None and ground_truth is not None:
        try:
            if abs(pred - float(ground_truth)) < 1e-6:
                is_correct = True
                correct_count += 1
        except (ValueError, TypeError):
            pass

    print(f"Sample {i+1}: GT={ground_truth}, Pred={pred}, Correct={is_correct}")

accuracy = correct_count / total if total > 0 else 0
print(f"\n✅ Accuracy: {accuracy:.2%} ({correct_count}/{total})")
print("Predictions:", evals)

Sample 1: GT=18.0, Pred=18.0, Correct=True
Sample 2: GT=45.0, Pred=15.0, Correct=False
Sample 3: GT=230.0, Pred=230.0, Correct=True
Sample 4: GT=26.0, Pred=26.0, Correct=True
Sample 5: GT=35.0, Pred=35.0, Correct=True
Sample 6: GT=8.0, Pred=8.0, Correct=True
Sample 7: GT=8.0, Pred=8.0, Correct=True
Sample 8: GT=3.0, Pred=3.0, Correct=True
Sample 9: GT=300.0, Pred=25.0, Correct=False
Sample 10: GT=221.0, Pred=221.0, Correct=True
Sample 11: GT=10.0, Pred=10.0, Correct=True
Sample 12: GT=8000.0, Pred=8000.0, Correct=True
Sample 13: GT=3.0, Pred=3.0, Correct=True
Sample 14: GT=500.0, Pred=500.0, Correct=True
Sample 15: GT=24.0, Pred=24.0, Correct=True
Sample 16: GT=240.0, Pred=240.0, Correct=True
Sample 17: GT=880.0, Pred=888.0, Correct=False
Sample 18: GT=6.0, Pred=6.0, Correct=True
Sample 19: GT=20.0, Pred=36.0, Correct=False
Sample 20: GT=4.0, Pred=4.0, Correct=True
Sample 21: GT=16.0, Pred=16.0, Correct=True
Sample 22: GT=18.0, Pred=18.0, Correct=True
Sample 23: GT=100.0, Pred=60.0, Co